# IONS-X Deep Emergence Lab - Guided Quickstart

This notebook walks through the four moving parts of the simulation - the **Target** field, the **Operators** (agents), the **Moderators** (environment), and the **emergent graph** - using quick settings so it runs top to bottom in a minute or two on CPU.

> It runs the same code as `ions_x_deep_emergence.py`; nothing here is a mockup.


## 0. Setup

Install dependencies if you are on a fresh environment (Colab: uncomment the pip line).


In [ ]:
# !pip install -q numpy scipy matplotlib networkx pandas pillow ipython
import importlib.util, json
from pathlib import Path
import numpy as np

# Load the single-module lab from the repo root (works in-repo and in Colab after clone).
module_path = Path('..') / 'ions_x_deep_emergence.py'
if not module_path.exists():
    module_path = Path('ions_x_deep_emergence.py')
spec = importlib.util.spec_from_file_location('ions_x_deep_emergence', module_path)
sim = importlib.util.module_from_spec(spec); spec.loader.exec_module(sim)
print('Backend:', 'GPU' if sim.on_gpu else 'CPU')


## 1. Targets - the 4-channel field

The Target is a 2D field with four channels: EM/RF, optical/IR, a consciousness proxy, and a control baseline. In synthetic mode the field evolves through spectral diffusion. Let's look at the spatial basis each channel starts from.


In [ ]:
import matplotlib.pyplot as plt
bases = sim._spatial_bases(64)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, basis, name in zip(axes, bases, sim.CHANNEL_NAMES):
    ax.imshow(basis, cmap='magma'); ax.set_title(name, fontsize=10); ax.axis('off')
plt.tight_layout(); plt.show()


## 2. Operators - autonomous sampling agents

Operators walk the field, keep a short memory of what they sample, and report channel pairs whose correlation exceeds a threshold. Here is a single operator discovering a correlated pair from a small synthetic memory.


In [ ]:
agent = sim.Agent(0, 'perceiver')
rng = np.random.RandomState(1)
for _ in range(60):
    x = rng.normal()
    agent.observe(sim.Observation(values=(x, x, rng.normal(), rng.normal()), env_factor=1.0))
discoveries = agent.discover(threshold=0.5)
discoveries


## 3. Moderators - the changing environment

Moderators modulate field evolution and the discovery threshold over time. The synthetic `EnvironmentalModerators` adds slow periodic drift plus short coherence windows; the empirical `RealWorldModerator` derives pressure from covariates (Kp index, lunar phase, sidereal time, X-ray flux).


In [ ]:
env = sim.EnvironmentalModerators()
series = []
for t in range(200):
    env.update(t); series.append(env.get_modulation(t))
plt.figure(figsize=(10, 2.8))
plt.plot(series); plt.title('Environmental coherence factor over time')
plt.xlabel('frame'); plt.ylabel('modulation'); plt.show()


## 4. Run a quick simulation and read the metrics sidecar

Run the whole thing with the `quick` experiment preset and a small frame budget. This writes an HTML animation and a `.metrics.json` summary next to it.


In [ ]:
result = sim.main(['--experiment', 'quick', '--frames', '60', '--agents', '60',
                   '--field-res', '48', '--output', 'outputs/notebook_demo.html'])
summary = json.loads(Path(result.summary_path).read_text())
summary


## 5. View the animation

Open `outputs/notebook_demo.html` in a browser, or display it inline below.


In [ ]:
from IPython.display import HTML
HTML(Path(result.output_path).read_text(encoding='utf-8'))


## Where to go next

- Try `--experiment arv`, `coherence`, or `dense-agents`.
- Feed real telemetry: `sim.main(['--preset', 'empirical', '--input-data', 'your.csv'])`.
- Calibrate a control threshold with `--preset baseline`.

Treat every output as a simulation artifact for exploring hypotheses - not a scientific claim on its own.
